In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import faiss
import time
import math
from pathlib import Path

CKPT_PATH = Path.home() / "projects" / "recsys" / "checkpoints" / "two_tower.pt"
PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

# rebuild model class (must match training-time definition)
class TwoTower(nn.Module):
    def __init__(self, n_users, n_movies, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_movies, dim)
    def encode_user(self, u): return self.user_emb(u)
    def encode_item(self, m): return self.item_emb(m)


# load checkpoint
print(f"\nloading checkpoint: {CKPT_PATH}")
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

n_users = ckpt["n_users"]
n_movies = ckpt["n_movies"]
emb_dim = ckpt["config"]["emb_dim"]
user_to_idx = ckpt["user_to_idx"]
movie_to_idx = ckpt["movie_to_idx"]

print(f"  n_users:   {n_users:,}")
print(f"  n_movies:  {n_movies:,}")
print(f"  emb_dim:   {emb_dim}")
print(f"  best_epoch: {ckpt['best_epoch']}")
print(f"\n  metrics from training:")
for k, v in ckpt["metrics"].items():
    print(f"    {k}: {v:.4f}")

model = TwoTower(n_users, n_movies, dim=emb_dim).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"\nmodel loaded, params: {sum(p.numel() for p in model.parameters()):,}")

device: mps

loading checkpoint: /Users/nitishpatil/projects/recsys/checkpoints/two_tower.pt
  n_users:   150,330
  n_movies:  45,058
  emb_dim:   64
  best_epoch: 5

  metrics from training:
    recall@5: 0.0258
    recall@10: 0.0435
    recall@20: 0.0694
    ndcg@5: 0.1154
    ndcg@10: 0.1051
    ndcg@20: 0.1028
    pop_recall@5: 0.0202
    pop_recall@10: 0.0301
    pop_recall@20: 0.0468

model loaded, params: 12,504,832


In [2]:
# extract all item embeddings as a numpy array
print("extracting item embeddings...")
t0 = time.time()
with torch.no_grad():
    all_movies = torch.arange(n_movies, dtype=torch.long, device=device)
    item_vecs = model.encode_item(all_movies).cpu().numpy().astype(np.float32)
print(f"  shape: {item_vecs.shape}, dtype: {item_vecs.dtype}, {time.time()-t0:.2f}s")

# build faiss index — IndexFlatIP = exact inner product, brute force
# (exact for small catalogs; switch to IVF/HNSW at >1m items)
print(f"\nbuilding faiss IndexFlatIP (dim={emb_dim})...")
t0 = time.time()
index = faiss.IndexFlatIP(emb_dim)
index.add(item_vecs)
print(f"  built in {time.time()-t0:.3f}s, ntotal: {index.ntotal:,}")

# memory footprint
size_mb = item_vecs.nbytes / 1e6
print(f"  index memory: {size_mb:.1f} mb ({n_movies:,} × {emb_dim} × 4 bytes float32)")

# sanity check: search for movie 0 — should return itself with highest similarity
print("\nsanity check: nearest neighbors of movie_idx=0")
query = item_vecs[0:1]   # shape (1, dim)
D, I = index.search(query, k=5)
print(f"  query shape: {query.shape}")
print(f"  results — distances: {D[0]}")
print(f"  results — indices:   {I[0]}")
print(f"  is movie 0 first? {I[0][0] == 0}")

extracting item embeddings...
  shape: (45058, 64), dtype: float32, 0.04s

building faiss IndexFlatIP (dim=64)...
  built in 0.003s, ntotal: 45,058
  index memory: 11.5 mb (45,058 × 64 × 4 bytes float32)

sanity check: nearest neighbors of movie_idx=0
  query shape: (1, 64)
  results — distances: [0.6736826  0.653653   0.65174335 0.62350816 0.62295794]
  results — indices:   [2180 3158  272  407 2184]
  is movie 0 first? False


In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import numpy as np
import faiss
import time
from pathlib import Path

faiss.omp_set_num_threads(1)
torch.set_num_threads(1)

CKPT_PATH = Path.home() / "projects" / "recsys" / "checkpoints" / "two_tower.pt"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

class TwoTower(nn.Module):
    def __init__(self, n_users, n_movies, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_movies, dim)
    def encode_user(self, u): return self.user_emb(u)
    def encode_item(self, m): return self.item_emb(m)

print("imports + threading set")

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
n_users = ckpt["n_users"]
n_movies = ckpt["n_movies"]
emb_dim = ckpt["config"]["emb_dim"]
model = TwoTower(n_users, n_movies, dim=emb_dim).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"model loaded: {n_users:,} users × {n_movies:,} movies × {emb_dim} dim")

with torch.no_grad():
    item_vecs = model.encode_item(torch.arange(n_movies, dtype=torch.long, device=device)).cpu().numpy().astype(np.float32)
print(f"item_vecs shape: {item_vecs.shape}, dtype: {item_vecs.dtype}")

index = faiss.IndexFlatIP(emb_dim)
index.add(item_vecs)
print(f"index built. ntotal: {index.ntotal:,}")

D, I = index.search(item_vecs[0:1], 5)
print(f"query 0 top-5: distances {D[0]}, indices {I[0]}")

imports + threading set
model loaded: 150,330 users × 45,058 movies × 64 dim
item_vecs shape: (45058, 64), dtype: float32
index built. ntotal: 45,058
query 0 top-5: distances [0.6736826  0.653653   0.65174335 0.62350816 0.62295794], indices [2180 3158  272  407 2184]


In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import faiss
import time
import math
from pathlib import Path

faiss.omp_set_num_threads(1)
torch.set_num_threads(1)

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
CKPT_PATH = Path.home() / "projects" / "recsys" / "checkpoints" / "two_tower.pt"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


class TwoTower(nn.Module):
    def __init__(self, n_users, n_movies, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_movies, dim)
    def encode_user(self, u): return self.user_emb(u)
    def encode_item(self, m): return self.item_emb(m)


# load checkpoint
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
n_users = ckpt["n_users"]
n_movies = ckpt["n_movies"]
emb_dim = ckpt["config"]["emb_dim"]
user_to_idx = ckpt["user_to_idx"]
movie_to_idx = ckpt["movie_to_idx"]

model = TwoTower(n_users, n_movies, dim=emb_dim).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# extract item embeddings + build faiss index
with torch.no_grad():
    item_vecs_t = model.encode_item(torch.arange(n_movies, dtype=torch.long, device=device))
    item_vecs = item_vecs_t.cpu().numpy().astype(np.float32)
print(f"item embeddings extracted: {item_vecs.shape}")

index = faiss.IndexFlatIP(emb_dim)
index.add(item_vecs)
print(f"faiss index built: {index.ntotal:,} items, dim={emb_dim}")

# load data + setup val sample
ratings = pd.read_parquet(PARQUET_DIR / "ratings_clean.parquet")
cutoff_ts = ratings["timestamp"].quantile(0.9)
train_df = ratings[ratings["timestamp"] < cutoff_ts].copy()
val_df = ratings[ratings["timestamp"] >= cutoff_ts].copy()

for df in (train_df, val_df):
    df["user_idx"] = df["userId"].map(user_to_idx)
    df["movie_idx"] = df["movieId"].map(movie_to_idx)
    df.dropna(subset=["user_idx", "movie_idx"], inplace=True)
    df["user_idx"] = df["user_idx"].astype(np.int32)
    df["movie_idx"] = df["movie_idx"].astype(np.int32)

# precompute per-user seen sets + per-user liked-in-val
train_by_user = train_df.groupby("user_idx")["movie_idx"].apply(set)
val_by_user_liked = val_df[val_df["rating"] >= 4.0].groupby("user_idx")["movie_idx"].apply(set)
eligible = list(val_by_user_liked.index)
print(f"eligible val users: {len(eligible):,}")

rng = np.random.RandomState(42)
sample_users = rng.choice(eligible, size=1000, replace=False)
print(f"sample size: {len(sample_users)}\n")


def _dcg(rels):
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))


# ---------- benchmark 1: naive (per-user matmul on mps, same as day 11) ----------
print("=== naive: per-user matmul on mps ===")
metrics_naive = {f"recall@{k}": [] for k in (5, 10, 20)}
metrics_naive.update({f"ndcg@{k}": [] for k in (5, 10, 20)})

t0 = time.time()
with torch.no_grad():
    for user_idx in sample_users:
        seen = train_by_user.get(user_idx, set())
        liked = val_by_user_liked[user_idx]
        mask = np.ones(n_movies, dtype=bool)
        mask[list(seen)] = False
        user_t = torch.tensor([int(user_idx)], dtype=torch.long, device=device)
        user_vec = model.encode_user(user_t)
        scores = (item_vecs_t @ user_vec.T).squeeze(1).cpu().numpy()
        scores[~mask] = -np.inf
        for k in (5, 10, 20):
            top_k = np.argpartition(-scores, k)[:k]
            top_sorted = top_k[np.argsort(-scores[top_k])]
            hits = liked.intersection(top_sorted.tolist())
            metrics_naive[f"recall@{k}"].append(len(hits) / len(liked))
            rels = [1 if mid in liked else 0 for mid in top_sorted]
            ideal = [1] * min(k, len(liked))
            ndcg = _dcg(rels) / _dcg(ideal) if ideal else 0
            metrics_naive[f"ndcg@{k}"].append(ndcg)
naive_time = time.time() - t0
print(f"  time: {naive_time:.2f}s ({1000/naive_time:.0f} users/sec)")
for k in (5, 10, 20):
    print(f"  recall@{k}: {np.mean(metrics_naive[f'recall@{k}']):.4f}")
print()


# ---------- benchmark 2: faiss batched ----------
print("=== faiss: batched query ===")
metrics_faiss = {f"recall@{k}": [] for k in (5, 10, 20)}
metrics_faiss.update({f"ndcg@{k}": [] for k in (5, 10, 20)})

K_OVERFETCH = 200  # fetch top-200 to leave headroom after masking seen items

t0 = time.time()
# encode all sample users in one shot on mps
with torch.no_grad():
    sample_users_t = torch.tensor(sample_users.astype(np.int64), device=device)
    user_vecs_all = model.encode_user(sample_users_t).cpu().numpy().astype(np.float32)
encode_time = time.time() - t0
print(f"  encoded {len(sample_users)} user vectors: {encode_time:.3f}s")

# single batched search
t1 = time.time()
D_all, I_all = index.search(user_vecs_all, K_OVERFETCH)
search_time = time.time() - t1
print(f"  faiss batched search ({K_OVERFETCH} per user): {search_time:.3f}s")

# post-process: mask seen items, compute metrics
t2 = time.time()
for i, user_idx in enumerate(sample_users):
    seen = train_by_user.get(user_idx, set())
    liked = val_by_user_liked[user_idx]
    raw_top = I_all[i]
    filtered = [m for m in raw_top if m not in seen]
    for k in (5, 10, 20):
        top_k = filtered[:k]
        hits = liked.intersection(top_k)
        metrics_faiss[f"recall@{k}"].append(len(hits) / len(liked))
        rels = [1 if mid in liked else 0 for mid in top_k]
        ideal = [1] * min(k, len(liked))
        ndcg = _dcg(rels) / _dcg(ideal) if ideal else 0
        metrics_faiss[f"ndcg@{k}"].append(ndcg)
postproc_time = time.time() - t2
print(f"  post-processing (mask + metrics): {postproc_time:.3f}s")
faiss_total = encode_time + search_time + postproc_time
print(f"  total: {faiss_total:.3f}s ({1000/faiss_total:.0f} users/sec)")
for k in (5, 10, 20):
    print(f"  recall@{k}: {np.mean(metrics_faiss[f'recall@{k}']):.4f}")

print(f"\nspeedup: {naive_time/faiss_total:.1f}x")
print(f"recall@10 match? naive={np.mean(metrics_naive['recall@10']):.4f}, "
      f"faiss={np.mean(metrics_faiss['recall@10']):.4f}")

item embeddings extracted: (45058, 64)
faiss index built: 45,058 items, dim=64
eligible val users: 6,282
sample size: 1000

=== naive: per-user matmul on mps ===
  time: 1.51s (662 users/sec)
  recall@5: 0.0258
  recall@10: 0.0435
  recall@20: 0.0694

=== faiss: batched query ===
  encoded 1000 user vectors: 0.007s
  faiss batched search (200 per user): 0.043s
  post-processing (mask + metrics): 0.019s
  total: 0.068s (14726 users/sec)
  recall@5: 0.0258
  recall@10: 0.0435
  recall@20: 0.0691

speedup: 22.3x
recall@10 match? naive=0.0435, faiss=0.0435


integrated faiss for batched retrieval. naive per-user mps matmul: 662 users/sec. faiss IndexFlatIP with batched query: 14,726 users/sec (22.3x speedup) with bit-identical recall@10. at our 45k-item scale IndexFlatIP is exact; for 100m+ catalogs the same code path swaps in IndexHNSWFlat for sublinear approximate search with controlled recall loss. this is the production retrieval pattern: embeddings precomputed offline, ann index queried online.